# Combine benchmark records → EVIDENCE.md

Upload the `results_<tag>_bs<N>.json` files produced by `benchmark.ipynb` (run once per
batch size). This reads them, writes `EVIDENCE.md` (prose + tables), and shows the
comparison tables. No model is loaded — a pure merge of the saved records, so it runs
anywhere (CPU is fine).

## 0 · Setup (tooling only)

In [ ]:
import os, sys, getpass, logging, subprocess
logging.basicConfig(level=logging.INFO, format="%(message)s")
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pandas"], check=False)
    REPO = "/content/repo"
    if not os.path.isdir(os.path.join(REPO, "Embodied")):
        tok = getpass.getpass("GitHub token (Contents: read): ").strip()
        res = subprocess.run(
            ["git", "clone", "-b", "batched-vectorized-decode",
             f"https://{tok}@github.com/semajyllek/Eagle.git", REPO],
            capture_output=True, text=True, env={**os.environ})
        if res.returncode != 0:
            raise RuntimeError((res.stderr or res.stdout).replace(tok, "***"))
else:
    REPO = subprocess.run(["git", "rev-parse", "--show-toplevel"],
                          capture_output=True, text=True).stdout.strip()
sys.path.insert(0, os.path.join(REPO, "Embodied"))

## 1 · Upload the per-branch records

In [ ]:
paths = []
try:
    from google.colab import files
    up = files.upload()                 # select results_batched_bs8.json, results_batched_bs16.json, ...
    paths = list(up.keys())
except Exception:
    import glob
    paths = sorted(glob.glob("results_*.json"))
print("records:", paths)

## 2 · Combine → EVIDENCE.md + comparison tables

In [ ]:
from IPython.display import Markdown, display
from repro.combine import (load_results, write_doc, detection_table,
                           length_buckets_table, grounded_table, speed_plot,
                           compile_loop_table, compile_ap_table, stacked_eval_table)
res = load_results(paths)
write_doc(res, "EVIDENCE.md")
print("Detection: batched speedup + parity"); display(Markdown(detection_table(res)))
print("Throughput sweep (sequential vs batched)"); display(Markdown(speed_plot(res)))
display(Markdown(length_buckets_table(res)))
print("One image, many queries"); display(Markdown(grounded_table(res)))
print("Real decode-loop torch.compile A/B"); display(Markdown(compile_loop_table(res)))
print("Compile mAP gate"); display(Markdown(compile_ap_table(res)))
print("End-to-end stacked speedup (vs pristine)"); display(Markdown(stacked_eval_table(res)))
try:
    from google.colab import files; files.download("EVIDENCE.md")
except Exception:
    print("wrote EVIDENCE.md")